Multi-armed Bandits Hyper-parameter tuning

In [14]:
import torch
import math, os
import torch.nn as nn
import torch.optim as optim
from dataclasses import dataclass
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset

Config class

In [15]:
@dataclass
class ParameterConfig:
    learning_rate: float
    batch_size: int
    hidden_layers: list[int]

In [16]:
CONFIGURATIONS: list[ParameterConfig] = [
    ParameterConfig(learning_rate=0.1, batch_size=64, hidden_layers=[128]),
    ParameterConfig(learning_rate=0.01, batch_size=128, hidden_layers=[128, 64]),
    ParameterConfig(learning_rate=0.001, batch_size=32, hidden_layers=[64]),
    ParameterConfig(learning_rate=0.05, batch_size=32, hidden_layers=[256, 128]),
    ParameterConfig(learning_rate=0.005, batch_size=64, hidden_layers=[64, 32]),
    ParameterConfig(learning_rate=0.2, batch_size=256, hidden_layers=[512]),
    ParameterConfig(learning_rate=0.02, batch_size=16, hidden_layers=[128, 128]),
    ParameterConfig(learning_rate=0.0005, batch_size=128, hidden_layers=[32]),
    ParameterConfig(learning_rate=0.15, batch_size=96, hidden_layers=[256]),
    ParameterConfig(learning_rate=0.002, batch_size=48, hidden_layers=[64, 64])
]

Params

In [17]:
input_dim = 28 * 28 # MNIST input dimension
output_dim = 10 # MNIST output class
K_arms = len(CONFIGURATIONS)
global_epoch_budget = 20
exploration_constant = 0.15
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Persistent Storage Arrays

In [18]:
active_models = [None] * len(CONFIGURATIONS)
active_optimizers = [None] * len(CONFIGURATIONS)

Dynamically build the Trainable Network

In [19]:
def build_neural_network(config: ParameterConfig) -> nn.Sequential:
    layers = nn.Sequential()
    init_dim = input_dim

    for hidden_dim in config.hidden_layers:
        layers.append(nn.Linear(init_dim, hidden_dim))
        layers.append(nn.ReLU())
        init_dim = hidden_dim

    layers.append(nn.Linear(init_dim, output_dim))
    return layers

Testing the function

In [20]:
testmodel = build_neural_network(CONFIGURATIONS[0])
testmodel

Sequential(
  (0): Linear(in_features=784, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=10, bias=True)
)

Preparing the dataset

In [21]:
transform_pipeline = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_set = datasets.MNIST(root='/home/bukunmi/ml-journey/datasets/data', train=True, download=True, transform=transform_pipeline)
test_set = datasets.MNIST(root='/home/bukunmi/ml-journey/datasets/data', train=False, download=True, transform=transform_pipeline)

Bandit Selection Engine

In [22]:
class UCBHyperparameterTuning: # Upper Confidence Bound for Bandits
    def __init__(self, num_configs: int, total_budget: int, c: float = 2):
        self.K = num_configs
        self.max_t = total_budget
        self.c = c # Exploration Constant
        self.t = 0 # Global counter for total epochs
        self.N = [0] * self.K # Array tracking total epochs per config
        self.X_bar = [0] * self.K # Array tracking running means of validation accuracies
        self.scores = [0] * self.K # Array tracking latest evaluated UCB values

    def select_best_config_index(self) -> int:
        max_score = float("-inf")
        selected_index = 0
        log_t = math.log(self.t) if self.t > 0 else 0.0
        for i in range(self.K):
            if self.N[i] == 0:
                current_score = float("inf")
            else:
                exploitation = self.X_bar[i]
                exploration = self.c * math.sqrt(log_t / self.N[i])
                current_score = exploitation + exploration
            self.scores[i] = current_score
            if self.scores[i] > max_score:
                max_score = self.scores[i]
                selected_index = i
        return selected_index

    def update_state(self, chosen_index: int, reward: float):
        self.N[chosen_index] += 1
        self.t += 1

        delta = reward - self.X_bar[chosen_index]
        self.X_bar[chosen_index] += delta / self.N[chosen_index]

Trigger Training Epochs

In [23]:
def trigger_training_epoch(config_index: int, train_dataset, val_dataset, device) -> float:
    config = CONFIGURATIONS[config_index]

    if active_models[config_index] is None:
        model = build_neural_network(config).to(device)
        optimizer = optim.SGD(model.parameters(), lr=config.learning_rate)

        active_models[config_index] = model
        active_optimizers[config_index] = optimizer
    else:
        model = active_models[config_index]
        optimizer = active_optimizers[config_index]

    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        images = images.view(images.size(0), -1)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            images = images.view(images.size(0), -1)
            logits = model(images)
            _, pred = torch.max(logits.data, 1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
    return correct / total

Instantiating Bandits

In [24]:
bandit = UCBHyperparameterTuning(num_configs=K_arms, total_budget=global_epoch_budget, c=exploration_constant)

Running Bandits initial Seeding

In [25]:
for idx in range(bandit.K):
    epoch_accuracy = trigger_training_epoch(idx, train_set, test_set, device)
    bandit.N[idx] = 1
    bandit.X_bar[idx] = epoch_accuracy
    bandit.t += 1
    print(f"Config {idx} initialized -> Validation Accuracy: {epoch_accuracy:.4f} | Total Pull Distribution: {bandit.N}")

Config 0 initialized -> Validation Accuracy: 0.9588 | Total Pull Distribution: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Config 1 initialized -> Validation Accuracy: 0.8764 | Total Pull Distribution: [1, 1, 0, 0, 0, 0, 0, 0, 0, 0]
Config 2 initialized -> Validation Accuracy: 0.8553 | Total Pull Distribution: [1, 1, 1, 0, 0, 0, 0, 0, 0, 0]
Config 3 initialized -> Validation Accuracy: 0.9570 | Total Pull Distribution: [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
Config 4 initialized -> Validation Accuracy: 0.8761 | Total Pull Distribution: [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]
Config 5 initialized -> Validation Accuracy: 0.9463 | Total Pull Distribution: [1, 1, 1, 1, 1, 1, 0, 0, 0, 0]
Config 6 initialized -> Validation Accuracy: 0.9571 | Total Pull Distribution: [1, 1, 1, 1, 1, 1, 1, 0, 0, 0]
Config 7 initialized -> Validation Accuracy: 0.4153 | Total Pull Distribution: [1, 1, 1, 1, 1, 1, 1, 1, 0, 0]
Config 8 initialized -> Validation Accuracy: 0.9643 | Total Pull Distribution: [1, 1, 1, 1, 1, 1, 1, 1, 1, 0]
Config 9 i

Live Optimization Loop

In [26]:
while bandit.t < bandit.max_t:
    chosen_config_index = bandit.select_best_config_index()
    epoch_accuracy = trigger_training_epoch(chosen_config_index, train_set, test_set, device)
    bandit.update_state(chosen_config_index, epoch_accuracy)
    print(f"Global Time Step: {bandit.t:02d} | Picked Config: {chosen_config_index} | "
        f"Current Accuracy: {epoch_accuracy:.4f} | Updated Mean Accuracies: {[round(x, 3) for x in bandit.X_bar]} | "
        f"Allocated Pull Counts: {bandit.N}")

winning_index = bandit.X_bar.index(max(bandit.X_bar))

Global Time Step: 11 | Picked Config: 8 | Current Accuracy: 0.9723 | Updated Mean Accuracies: [0.959, 0.876, 0.855, 0.957, 0.876, 0.946, 0.957, 0.415, 0.968, 0.809] | Allocated Pull Counts: [1, 1, 1, 1, 1, 1, 1, 1, 2, 1]
Global Time Step: 12 | Picked Config: 0 | Current Accuracy: 0.9666 | Updated Mean Accuracies: [0.963, 0.876, 0.855, 0.957, 0.876, 0.946, 0.957, 0.415, 0.968, 0.809] | Allocated Pull Counts: [2, 1, 1, 1, 1, 1, 1, 1, 2, 1]
Global Time Step: 13 | Picked Config: 6 | Current Accuracy: 0.9699 | Updated Mean Accuracies: [0.963, 0.876, 0.855, 0.957, 0.876, 0.946, 0.964, 0.415, 0.968, 0.809] | Allocated Pull Counts: [2, 1, 1, 1, 1, 1, 2, 1, 2, 1]
Global Time Step: 14 | Picked Config: 3 | Current Accuracy: 0.9749 | Updated Mean Accuracies: [0.963, 0.876, 0.855, 0.966, 0.876, 0.946, 0.964, 0.415, 0.968, 0.809] | Allocated Pull Counts: [2, 1, 1, 2, 1, 1, 2, 1, 2, 1]
Global Time Step: 15 | Picked Config: 5 | Current Accuracy: 0.9653 | Updated Mean Accuracies: [0.963, 0.876, 0.855, 

In [27]:
print(f"Winning Configuration Index: {winning_index}")
print(f"Winning Parameter Settings : {CONFIGURATIONS[winning_index]}")
print(f"Final Step Resource Allocation Allocation Matrix : {bandit.N}")
print(f"Final Convergence Estimated Mean Validation Accuracies: {[round(x, 4) for x in bandit.X_bar]}")

Winning Configuration Index: 8
Winning Parameter Settings : ParameterConfig(learning_rate=0.15, batch_size=96, hidden_layers=[256])
Final Step Resource Allocation Allocation Matrix : [3, 1, 1, 3, 1, 3, 3, 1, 3, 1]
Final Convergence Estimated Mean Validation Accuracies: [0.9641, 0.8764, 0.8553, 0.9694, 0.8761, 0.9605, 0.9679, 0.4153, 0.9698, 0.8092]
